# Other embedders

In [1]:
import lancedb
import pandas as pd 

db = lancedb.connect("knowledge_base")

df = pd.read_json("data/jokes.json")
df.head()

,jokes
0,Parallel lines have so much in common—it’s sad...
1,"ETL stands for “Extract, Transform, Leave for ..."
2,What do you call a snake that runs your script...
3,"Gold walks into a bar. The bartender says, “Au..."
4,C# devs don’t argue; they just throw exceptions.


## Sentence transformer 

In [2]:
from lancedb.embeddings import get_registry
from lancedb.pydantic import LanceModel, Vector

model = (
    get_registry()
    .get("sentence-transformers")
    .create(name="intfloat/multilingual-e5-small", device="cpu")
)

class JokeModel(LanceModel):
    jokes: str = model.SourceField() 
    vector: Vector(model.ndims()) = model.VectorField()  


print(f"{model.ndims() = }")

table = db.create_table("jokes_sentence_transformers", schema=JokeModel)
table

/Users/juliusfaltskoglund/dev/github/AI_OPS_julius_falt_skoglund_DE25/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1112 13:20:49.592000 93053 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 23000.54it/s]


model.ndims() = 384


LanceTable(name='jokes_sentence_transformers', _conn=LanceDBConnection(uri='/Users/juliusfaltskoglund/dev/github/AI_OPS_julius_falt_skoglund_DE25/code-alongs/09_lancedb/knowledge_base'))

In [5]:
table.add(df)

AddResult(version=2)

In [6]:
table.search().limit(5).to_pandas()

,jokes,vector
0,Parallel lines have so much in common—it’s sad...,"[0.037198, -0.019918, -0.051841, -0.032693, 0...."
1,"ETL stands for “Extract, Transform, Leave for ...","[0.050497, -0.006426, -0.071011, -0.027726, 0...."
2,What do you call a snake that runs your script...,"[0.073168, -0.00455, -0.027777, -0.054148, 0.0..."
3,"Gold walks into a bar. The bartender says, “Au...","[0.045366, -0.05297, -0.066901, -0.02893, 0.11..."
4,C# devs don’t argue; they just throw exceptions.,"[0.028011, 0.004646, -0.057664, -0.064801, 0.0..."


In [7]:
table.search("kemiskämt").to_pandas().head()

,jokes,vector,_distance
0,Why did the chemist ground his kids? Because t...,"[-0.000735, -0.029147, -0.051178, -0.053038, 0...",0.388733
1,Chemists have all the solutions… mostly in bea...,"[0.025907, -0.022183, -0.06204, -0.036586, 0.0...",0.391546
2,Oxygen and Magnesium got together—OMg!,"[0.050211, -0.010728, -0.099408, -0.035509, 0....",0.395784
3,I told a chemistry joke… there was no reaction.,"[0.048405, 0.011754, -0.043688, -0.029928, 0.1...",0.417908
4,Never trust an atom—they make up everything.,"[0.059282, -0.023624, -0.089997, -0.071139, 0....",0.443533
